In [0]:
# Cell 1: Load the IBM HR dataset 


import pandas as pd
from pyspark.sql import SparkSession


# verify it's running
print("Spark version:", spark.version)
print("Databricks is running!")

# sample HR data 
data = [
    (1, 41, "Sales", "Yes", 1102, 1, "Travel_Rarely", 2),
    (2, 49, "Research & Development", "No", 279, 8, "Travel_Frequently", 1),
    (3, 37, "Research & Development", "Yes", 1373, 2, "Travel_Rarely", 2),
    (4, 33, "Research & Development", "No", 1392, 3, "Travel_Frequently", 4),
    (5, 27, "Research & Development", "No", 591, 2, "Travel_Rarely", 1),
    (6, 32, "Research & Development", "No", 1005, 2, "Travel_Rarely", 2),
    (7, 59, "Research & Development", "No", 1324, 3, "Travel_Rarely", 3),
    (8, 30, "Sales", "Yes", 1389, 24, "Travel_Frequently", 1),
    (9, 38, "Research & Development", "No", 846, 23, "Travel_Rarely", 3),
    (10, 36, "Research & Development", "No", 1218, 27, "Travel_Rarely", 3),
]

columns = ["EmployeeID", "Age", "Department", "Attrition", 
           "DailyRate", "DistanceFromHome", "BusinessTravel", "Education"]

# Create Spark DataFrame 
df = spark.createDataFrame(data, columns)

print("\nDataFrame created successfully!")
print("Number of rows:", df.count())
df.show()

Spark version: 4.1.0
Databricks is running!

DataFrame created successfully!
Number of rows: 10
+----------+---+--------------------+---------+---------+----------------+-----------------+---------+
|EmployeeID|Age|          Department|Attrition|DailyRate|DistanceFromHome|   BusinessTravel|Education|
+----------+---+--------------------+---------+---------+----------------+-----------------+---------+
|         1| 41|               Sales|      Yes|     1102|               1|    Travel_Rarely|        2|
|         2| 49|Research & Develo...|       No|      279|               8|Travel_Frequently|        1|
|         3| 37|Research & Develo...|      Yes|     1373|               2|    Travel_Rarely|        2|
|         4| 33|Research & Develo...|       No|     1392|               3|Travel_Frequently|        4|
|         5| 27|Research & Develo...|       No|      591|               2|    Travel_Rarely|        1|
|         6| 32|Research & Develo...|       No|     1005|               2|    Tr

In [0]:
# Cell 2: Transform clean and engineer 
from pyspark.sql.functions import col, when, avg, count, round

# Convert Attrition to numeric flag
df_clean = df.withColumn("Attrition_Flag", 
                          when(col("Attrition") == "Yes", 1).otherwise(0))

# Create Age Group column
df_clean = df_clean.withColumn("AgeGroup",
    when(col("Age") < 26, "18-25")
    .when(col("Age") < 36, "26-35")
    .when(col("Age") < 46, "36-45")
    .when(col("Age") < 56, "46-55")
    .otherwise("56-65"))

# Create Distance Category
df_clean = df_clean.withColumn("DistanceCategory",
    when(col("DistanceFromHome") <= 5, "Near")
    .when(col("DistanceFromHome") <= 15, "Mid")
    .otherwise("Far"))

print("Transform complete!")
df_clean.show()

Transform complete!
+----------+---+--------------------+---------+---------+----------------+-----------------+---------+--------------+--------+----------------+
|EmployeeID|Age|          Department|Attrition|DailyRate|DistanceFromHome|   BusinessTravel|Education|Attrition_Flag|AgeGroup|DistanceCategory|
+----------+---+--------------------+---------+---------+----------------+-----------------+---------+--------------+--------+----------------+
|         1| 41|               Sales|      Yes|     1102|               1|    Travel_Rarely|        2|             1|   36-45|            Near|
|         2| 49|Research & Develo...|       No|      279|               8|Travel_Frequently|        1|             0|   46-55|             Mid|
|         3| 37|Research & Develo...|      Yes|     1373|               2|    Travel_Rarely|        2|             1|   36-45|            Near|
|         4| 33|Research & Develo...|       No|     1392|               3|Travel_Frequently|        4|             0

In [0]:
# Cell 3: Load into a Spark SQL 


df_clean.createOrReplaceTempView("hr_data")

print("Table registered! Now running Spark SQL queries...\n")

# Query 1: Attrition rate by department
print("=== Attrition by Department ===")
spark.sql("""
    SELECT Department,
           COUNT(*) as total_employees,
           SUM(Attrition_Flag) as left_count,
           ROUND(SUM(Attrition_Flag) * 100.0 / COUNT(*), 1) as attrition_rate
    FROM hr_data
    GROUP BY Department
    ORDER BY attrition_rate DESC
""").show()

# Query 2: Attrition by age group
print("=== Attrition by Age Group ===")
spark.sql("""
    SELECT AgeGroup,
           COUNT(*) as total,
           SUM(Attrition_Flag) as left_count,
           ROUND(SUM(Attrition_Flag) * 100.0 / COUNT(*), 1) as attrition_rate
    FROM hr_data
    GROUP BY AgeGroup
    ORDER BY attrition_rate DESC
""").show()

# Query 3: Average daily rate — leavers vs stayers
print("=== Daily Rate: Leavers vs Stayers ===")
spark.sql("""
    SELECT Attrition,
           ROUND(AVG(DailyRate), 0) as avg_daily_rate,
           COUNT(*) as count
    FROM hr_data
    GROUP BY Attrition
""").show()

Table registered! Now running Spark SQL queries...

=== Attrition by Department ===
+--------------------+---------------+----------+--------------+
|          Department|total_employees|left_count|attrition_rate|
+--------------------+---------------+----------+--------------+
|               Sales|              2|         2|         100.0|
|Research & Develo...|              8|         1|          12.5|
+--------------------+---------------+----------+--------------+

=== Attrition by Age Group ===
+--------+-----+----------+--------------+
|AgeGroup|total|left_count|attrition_rate|
+--------+-----+----------+--------------+
|   36-45|    4|         2|          50.0|
|   26-35|    4|         1|          25.0|
|   46-55|    1|         0|           0.0|
|   56-65|    1|         0|           0.0|
+--------+-----+----------+--------------+

=== Daily Rate: Leavers vs Stayers ===
+---------+--------------+-----+
|Attrition|avg_daily_rate|count|
+---------+--------------+-----+
|      Yes|

In [0]:
# Cell 4: Save as Delta table 

# Save as Delta table
df_clean.write.format("delta").mode("overwrite").saveAsTable("hr_analytics_delta")

print("✅ Delta table created!")
print("\nVerifying — reading back from Delta table:")

df_delta = spark.sql("SELECT * FROM hr_analytics_delta")
print("Rows in Delta table:", df_delta.count())

print("\n=== Delta Table History ===")
spark.sql("DESCRIBE HISTORY hr_analytics_delta").select(
    "version", "timestamp", "operation"
).show()

print("\n🎉 ETL Pipeline Complete!")
print("Data flowed: Raw → Transform → Spark SQL → Delta Lake")

✅ Delta table created!

Verifying — reading back from Delta table:
Rows in Delta table: 10

=== Delta Table History ===
+-------+-------------------+--------------------+
|version|          timestamp|           operation|
+-------+-------------------+--------------------+
|      0|2026-06-21 11:23:55|CREATE OR REPLACE...|
+-------+-------------------+--------------------+


🎉 ETL Pipeline Complete!
Data flowed: Raw → Transform → Spark SQL → Delta Lake
